In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve
)

In [2]:

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

print(df.shape)
print(df.head())


(5000, 9)
      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3          0  
4          1  


In [3]:

print(df.isnull().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [5]:

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print(df.isnull().sum())


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [6]:

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

print(df_train.shape, df_val.shape, df_test.shape)


(3000, 9) (1000, 9) (1000, 9)


In [7]:
for feature in numerical:
    auc = roc_auc_score(df_train["converted"], df_train[feature])

    if auc < 0.5:
        auc = roc_auc_score(df_train["converted"], -df_train[feature])

    print(feature, round(auc, 3))

annual_income 0.608
number_of_courses_viewed 0.723
interaction_count 0.765
lead_score 0.788


In [8]:
X_train = df_train.drop(columns=["converted"])
y_train = df_train["converted"]

X_val = df_val.drop(columns=["converted"])
y_val = df_val["converted"]

In [9]:
dv = DictVectorizer(sparse=False)

train_dicts = X_train.to_dict(orient="records")
val_dicts = X_val.to_dict(orient="records")

X_train_encoded = dv.fit_transform(train_dicts)
X_val_encoded = dv.transform(val_dicts)

print(X_train_encoded.shape)
print(X_val_encoded.shape)

(3000, 28)
(1000, 28)


In [10]:
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000
)

model.fit(X_train_encoded, y_train)

y_val_proba = model.predict_proba(X_val_encoded)[:, 1]

auc = roc_auc_score(y_val, y_val_proba)

print("Validation AUC:", round(auc, 3))

Validation AUC: 0.732


In [12]:
# calculating precision and recall high for which threshold
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0, 1.01, 0.01)

results = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)

    if precision != 0 or recall != 0:
        results.append({
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "difference": abs(precision - recall)
        })

results_df = pd.DataFrame(results)

print(results_df.loc[results_df["difference"].idxmin()])


threshold     0.630000
precision     0.732759
recall        0.725256
difference    0.007503
Name: 63, dtype: float64


In [13]:
# calculating f1
f1_results = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)

    if precision != 0 or recall != 0:
        f1 = f1_score(y_val, y_pred, zero_division=0)

        f1_results.append({
            "threshold": threshold,
            "f1": f1
        })

f1_df = pd.DataFrame(f1_results)

print(f1_df.loc[f1_df["f1"].idxmax()])


threshold    0.410000
f1           0.757616
Name: 41, dtype: float64


In [14]:
# Cross validation using 5 folds

from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=1)

auc_scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    y_fold_train = df_fold_train["converted"]
    y_fold_val = df_fold_val["converted"]

    dv_fold = DictVectorizer(sparse=False)

    train_dicts = df_fold_train.drop(columns=["converted"]).to_dict(orient="records")
    val_dicts = df_fold_val.drop(columns=["converted"]).to_dict(orient="records")

    X_fold_train = dv_fold.fit_transform(train_dicts)
    X_fold_val = dv_fold.transform(val_dicts)

    model_fold = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000
    )

    model_fold.fit(X_fold_train, y_fold_train)

    y_fold_proba = model_fold.predict_proba(X_fold_val)[:, 1]

    auc = roc_auc_score(y_fold_val, y_fold_proba)
    auc_scores.append(auc)

print("Fold AUC scores:", auc_scores)
print("Mean AUC:", round(np.mean(auc_scores), 3))
print("Standard deviation:", round(np.std(auc_scores), 3))


Fold AUC scores: [0.7305601027122453, 0.7295726148196622, 0.7363428071259397, 0.7412454717957564, 0.7209138590326581]
Mean AUC: 0.732
Standard deviation: 0.007


In [15]:
c_values = [0.000001, 0.001, 1]

for c in c_values:
    auc_scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        y_fold_train = df_fold_train["converted"]
        y_fold_val = df_fold_val["converted"]

        dv_fold = DictVectorizer(sparse=False)

        train_dicts = df_fold_train.drop(columns=["converted"]).to_dict(orient="records")
        val_dicts = df_fold_val.drop(columns=["converted"]).to_dict(orient="records")

        X_fold_train = dv_fold.fit_transform(train_dicts)
        X_fold_val = dv_fold.transform(val_dicts)

        model_fold = LogisticRegression(
            solver="liblinear",
            C=c,
            max_iter=1000
        )

        model_fold.fit(X_fold_train, y_fold_train)

        y_fold_proba = model_fold.predict_proba(X_fold_val)[:, 1]
        auc_scores.append(roc_auc_score(y_fold_val, y_fold_proba))

    print(
        f"C={c}: mean AUC={np.mean(auc_scores):.3f}, "
        f"std={np.std(auc_scores):.3f}"
    )

C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007
